![dvd_image](dvd_image.jpg)

Une entreprise de location de DVD a besoin de votre aide ! Elle souhaite estimer combien de jours un client va conserver un DVD en fonction de certaines caractéristiques et s’est tournée vers vous. Votre mission : tester quelques modèles de régression pour prédire le nombre de jours de location. L’entreprise vise un modèle dont la MSE sur un jeu de test est inférieure ou égale à 3. Le modèle que vous construirez aidera l’entreprise à mieux planifier ses stocks.

Les données sont fournies dans le fichier CSV `rental_info.csv`. Il contient les variables suivantes :
- `"rental_date"` : la date (et l’heure) à laquelle le client loue le DVD.
- `"return_date"` : la date (et l’heure) à laquelle le client rend le DVD.
- `"amount"` : le montant payé par le client pour la location du DVD.
- `"amount_2"` : le carré de `"amount"`.
- `"rental_rate"` : le tarif de location du DVD.
- `"rental_rate_2"` : le carré de `"rental_rate"`.
- `"release_year"` : l’année de sortie du film loué.
- `"length"` : la durée du film loué, en minutes.
- `"length_2"` : le carré de `"length"`.
- `"replacement_cost"` : le coût de remplacement du DVD pour l’entreprise.
- `"special_features"` : les bonus éventuels (par exemple, bandes‑annonces/scènes coupées) inclus sur le DVD.
- `"NC-17"`, `"PG"`, `"PG-13"`, `"R"` : variables indicatrices (dummies) de la classification du film. La valeur est 1 si le film a la classification indiquée par le nom de la colonne, sinon 0. Pour votre commodité, la variable de référence a déjà été supprimée.

In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error 
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv("rental_info.csv")

df["rental_length"] = pd.to_datetime(df["return_date"]) - pd.to_datetime(df["rental_date"])
df["rental_length_days"] = df["rental_length"].dt.days

df["deleted_scenes"] =  np.where(df["special_features"].str.contains("Deleted Scenes"), 1,0)
df["behind_the_scenes"] =  np.where(df["special_features"].str.contains("Behind the Scenes"), 1,0)

X = df.drop(["special_features", "rental_length", "rental_length_days", "rental_date", "return_date"], axis=1)
y = df["rental_length_days"]
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=9)

print(X.shape)
print(y.shape)
print(X.head())
print(y.head())
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)


(15861, 14)
(15861,)
   amount  release_year  ...  deleted_scenes  behind_the_scenes
0    2.99        2005.0  ...               0                  1
1    2.99        2005.0  ...               0                  1
2    2.99        2005.0  ...               0                  1
3    2.99        2005.0  ...               0                  1
4    2.99        2005.0  ...               0                  1

[5 rows x 14 columns]
0    3
1    2
2    7
3    2
4    4
Name: rental_length_days, dtype: int64
(12688, 14)
(3173, 14)
(12688,)
(3173,)


In [14]:
lasso = Lasso(alpha=0.3, random_state=9)
lasso.fit(X_train, y_train)
lasso_coef = lasso.coef_
X_lasso_train, X_lasso_test = X_train.iloc[:, lasso_coef > 0], X_test.iloc[:, lasso_coef > 0]

lasso_results = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": lasso.coef_
})

lasso_results = lasso_results.sort_values(
    by="Coefficient",
    ascending=False
)

print(lasso_results)

              Feature  Coefficient
0              amount     0.584104
9            amount_2     0.043622
10           length_2     0.000003
1        release_year     0.000000
2         rental_rate    -0.000000
3              length     0.000000
4    replacement_cost    -0.000000
5               NC-17     0.000000
6                  PG     0.000000
7               PG-13     0.000000
8                   R    -0.000000
12     deleted_scenes    -0.000000
13  behind_the_scenes     0.000000
11      rental_rate_2    -0.152984


In [15]:
linereg = LinearRegression()
linereg.fit(X_lasso_train, y_train)
y_pred = linereg.predict(X_lasso_test)
mse = mean_squared_error(y_test, y_pred)

print("Linear Regression MSE:", mse)

param_dist = {'n_estimators': np.arange(1,101,1),
          'max_depth':np.arange(1,11,1)}

rf = RandomForestRegressor()
rs = RandomizedSearchCV(rf,
                       param_distributions=param_dist,
                       cv=5,
                       random_state=9)
rs.fit(X_train, y_train)

print(rs.best_params_)
print(rs.best_score_)
print(rs.best_estimator_)



Linear Regression MSE: 4.812297241276244
{'n_estimators': 51, 'max_depth': 10}
0.6878113309829523
RandomForestRegressor(max_depth=10, n_estimators=51)


In [17]:
hyper_params = rs.best_params_

rf = RandomForestRegressor(n_estimators=hyper_params["n_estimators"], 
                           max_depth=hyper_params["max_depth"], 
                           random_state=9)
rf.fit(X_train,y_train)
rf_pred = rf.predict(X_test)
mse_random_forest= mean_squared_error(y_test, rf_pred)
print("Random Forest MSE:", mse_random_forest)

rmse_linear = np.sqrt(mse)
rmse_rf = np.sqrt(mse_random_forest)
print("Linear Regression RMSE:", rmse_linear)
print("Random Forest RMSE:", rmse_rf)

# Random forest gives lowest MSE so:
best_model = rf
best_mse = mse_random_forest

Random Forest MSE: 2.225667528098759
Linear Regression RMSE: 2.1936948833591794
Random Forest RMSE: 1.491867128164824
